# Results — model comparison (treatments 1–4)

The `hw-full-cli-sdk-skills` experiment: the **same FTI benchmark** run by four models, each across both
interfaces and both skill bundles.

| treatment | model |
|---|---|
| 1 | `claude-opus-4-8` (opus) |
| 2 | `mistral-large-3` |
| 3 | `claude-sonnet-4-6` (sonnet) |
| 4 | `mistral-medium-3-5` |

Every model ran all 26 tasks on **both interfaces** (`cli`, `sdk`) and **both skill bundles** (`none`,
`official`) — a 4 × 2 × 2 grid. Metrics:

- **completion** — fraction of the 26 tasks that produced a `valid` run (the agent finished and a deliverable was gradeable)
- **assert pass rate** — `asserts_passed / total_asserts`, over `valid` runs only
- **cost_usd** / **local_time_s** — mean over `valid` runs (lower is better)

This notebook is a hand-curated artifact; it is not regenerated by the runner.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

df = pd.read_csv("results.csv")
for c in ["asserts_passed", "total_asserts", "local_time_s", "cost_usd"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["valid"] = df["valid"].astype(str).str.lower() == "true"
df["pass_rate"] = df["asserts_passed"] / df["total_asserts"]

# Treatments 1-4 = the full model comparison (config order).
MODELS = [
    ("opus",           "1_hw-full-cli-sdk-skills-opus"),
    ("mistral-large",  "2_hw-full-cli-sdk-skills-mistral-large"),
    ("sonnet",         "3_hw-full-cli-sdk-skills-sonnet"),
    ("mistral-medium", "4_hw-full-cli-sdk-skills-mistral-medium"),
]
VARIANTS = [("cli", "none"), ("cli", "official"), ("sdk", "none"), ("sdk", "official")]

def cell(cfg, iface, sk):
    return df[(df.config == cfg) & (df.interface == iface) & (df.skills == sk)]

def metric(cfg, iface, sk, col):
    sub = cell(cfg, iface, sk)
    v = sub[sub.valid]
    if col == "completion":
        return v.shape[0] / sub.shape[0] if sub.shape[0] else float("nan")
    return v[col].mean()


## Grid: model × interface × skills

In [2]:
rows = []
for mlabel, cfg in MODELS:
    for iface, sk in VARIANTS:
        sub = cell(cfg, iface, sk)
        v = sub[sub.valid]
        rows.append({
            "model": mlabel, "interface": iface, "skills": sk,
            "valid": f"{v.shape[0]}/{sub.shape[0]}",
            "completion": v.shape[0] / sub.shape[0] if sub.shape[0] else float("nan"),
            "pass_rate": v.pass_rate.mean(),
            "cost_usd": v.cost_usd.mean(),
            "local_time_s": v.local_time_s.mean(),
        })
tbl = pd.DataFrame(rows).set_index(["model", "interface", "skills"])
tbl.style.format({"completion": "{:.0%}", "pass_rate": "{:.1%}",
                  "cost_usd": "{:.4f}", "local_time_s": "{:.0f}"})


## Charts

Four metrics, one bar per (interface, skills) variant. cli = grey, sdk = blue; lighter = no skills, darker = official skills.

In [3]:
METRIC_PLOTS = [
    ("completion",   "completion (valid / 26)",       "max", "pct"),
    ("pass_rate",    "assert pass rate (valid only)", "max", "pct"),
    ("cost_usd",     "cost (USD)",                    "min", "num"),
    ("local_time_s", "local time (s)",                "min", "num"),
]
COLORS = {("cli", "none"): "#b0bec5", ("cli", "official"): "#546e7a",
          ("sdk", "none"): "#64b5f6", ("sdk", "official"): "#1565c0"}
mlabels = [m for m, _ in MODELS]
x = np.arange(len(mlabels))
w = 0.2

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, (col, title, direction, kind) in zip(axes.ravel(), METRIC_PLOTS):
    for j, (iface, sk) in enumerate(VARIANTS):
        vals = [metric(cfg, iface, sk, col) for _, cfg in MODELS]
        bars = ax.bar(x + (j - 1.5) * w, vals, w, label=f"{iface}/{sk}", color=COLORS[(iface, sk)])
        for b, h in zip(bars, vals):
            if pd.notna(h):
                txt = f"{h:.0%}" if kind == "pct" else (f"{h:.3f}" if col == "cost_usd" else f"{h:.0f}")
                ax.annotate(txt, (b.get_x() + b.get_width() / 2, h),
                            ha="center", va="bottom", fontsize=6, rotation=90)
    ax.set_xticks(x)
    ax.set_xticklabels(mlabels, rotation=20, ha="right")
    ax.set_title(f"{title}  ({'higher' if direction == 'max' else 'lower'} = better)")
    ax.set_ylabel(title)
    if kind == "pct":
        ax.yaxis.set_major_formatter(PercentFormatter(1.0))
        ax.set_ylim(0, 1.08)
    ax.legend(fontsize=7, ncol=2, title="interface/skills")

fig.suptitle("Treatments 1–4 — model comparison across interface (cli/sdk) and skills (none/official)", y=1.01)
plt.tight_layout()
plt.show()


## Assertions: passed / total (valid runs, summed over the 26 tasks)

In [4]:
arows = []
for mlabel, cfg in MODELS:
    for iface, sk in VARIANTS:
        v = cell(cfg, iface, sk)
        v = v[v.valid]
        passed, total = v.asserts_passed.sum(), v.total_asserts.sum()
        arows.append({
            "model": mlabel, "interface": iface, "skills": sk,
            "passed": passed, "total": total,
            "rate": passed / total if total else float("nan"),
        })
adf = pd.DataFrame(arows).set_index(["model", "interface", "skills"])
adf.style.format({"passed": "{:.0f}", "total": "{:.0f}", "rate": "{:.1%}"})
